Membaca Data menggunakan elemen dari Pandas

In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/raw/ecommerce_raw_data.csv')
df.describe()

,phone,quantity,price,discount_pct,total_price,rating
count,1.433000e+03,1500.000000,1.500000e+03,1500.000000,1.500000e+03,1287.000000
mean,3.231072e+11,2.904667,4.448116e+06,14.820000,8.821599e+06,2.952603
std,3.100639e+11,3.576232,3.263626e+07,27.023862,5.834307e+07,1.479804
min,8.104043e+09,-1.000000,5.100000e+03,0.000000,-5.088886e+07,-1.000000
25%,8.604617e+09,1.000000,9.590000e+04,0.000000,1.461500e+05,2.000000
50%,6.281076e+11,2.000000,4.117000e+05,10.000000,6.437550e+05,3.000000
75%,6.285762e+11,3.000000,2.629350e+06,15.000000,4.015300e+06,4.000000
max,6.289977e+11,20.000000,7.610797e+08,197.000000,1.522159e+09,7.000000


Mengidentifikasi & menghapus duplikasi data 

In [21]:
numeric_cols = ['price', 'quantity', 'total_price', 'discount_pct', 'rating']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Penanganan terhadap kolom kosong (Missing Values), dengan menambahkan "unknown" pada kolom tersebut

In [22]:
#Pengecekan nilai yang hilang
missing_values = (df.isnull().sum()/len(df) * 100).sort_values(ascending=False)
print(missing_values[missing_values > 0])

#Kolom teks -> isi dengan 'Unknown'
df['email'] = df['email'].fillna('Unknown')
df['phone'] = df['phone'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')
df['category'] = df['category'].fillna('Unknown')
df['customer_name'] = df['customer_name'].fillna('Unknown')
# df['order_date'] = df['order_date'].fillna('NaN')

#Kolom kategorikal -> isi dengan modus
df['payment_method'] = df['payment_method'].fillna(df['payment_method'].mode()[0])
df['order_status'] = df['order_status'].fillna(df['order_status'].mode()[0])

#Kolom numerik -> isi dengan median (karena ada outlier)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating'] = df['rating'].fillna(df['rating'].median())

rating            14.200000
email              4.866667
phone              4.466667
order_status       3.333333
city               3.133333
payment_method     2.800000
category           1.733333
customer_name      1.400000
dtype: float64


Standarisasi Format
agar setiap data menjadi seragam sehingga lebih mudah untuk di olah

In [23]:
#Teks -> huruf kapital konsisten (title case/lower case)
df['city'] = df['city'].str.strip().str.title()
df['payment_method'] = df['payment_method'].str.strip().str.title()
df['order_status'] = df['order_status'].str.strip().str.title()
df['customer_name'] = df['customer_name'].str.strip().str.title()

#Mapping nilai tidak konsisten
payment_mapping = {
    'Transfer Bank' : 'Transfer Bank',
    'Cod' : 'COD',
    'Cash' : 'COD',
    'Cc' : 'Kartu Kredit',
    'Kartu Kredit' : 'Kartu Kredit',
    'Gopay' : 'GoPay',
    'Ovo' : 'OVO',
    'Qris' : 'QRIS'
}
df['payment_method'] = df['payment_method'].replace(payment_mapping)

#Status mapping
status_mapping = {
    'Selesai' : 'Selesai',
    'Success' : 'Selesai',
    'Sukses' : 'Selesai',
    'Batal' : 'Dibatalkan',
    'Cancel' : 'Dibatalkan',
    'Dibatalkan' : 'Dibatalkan',
    'Returned' : 'Dikembalikan',
    'Retur' : 'Dikembalikan'
}
df['order_status'] = df['order_status'].replace(status_mapping)

#Standarisasi Tanggal
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')


In [24]:
df = df.sort_values(by='order_date').reset_index(drop=True)

df['order_date'] = df['order_date'].fillna(method='ffill')

C:\Users\LEGION\AppData\Local\Temp\ipykernel_22276\2015515696.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['order_date'] = df['order_date'].fillna(method='ffill')


Validasi & Outlier

In [25]:
#Hapus kuantitas negatif
df['quantity'] = df['quantity'].clip(lower=1)

#Hapus diskon tidak logis
df['discount_pct'] = df['discount_pct'].clip(lower=0, upper=100)

#Hapus rating di luar 1-5
df['rating'] = df['rating'].clip(lower=1, upper=5).round(0)

#Deteksi Outlier menggunakan IQR
Q3 = df['price'].quantile(0.75)
Q1 = df['price'].quantile(0.25)
IQR = Q3 - Q1
df = df[df['price'].between(Q1 - 3 * IQR, Q3 + 3 * IQR)]

#MEnghitung ulang harga total setelah pembersihan
df['total_price'] = df['quantity'] * df['price'] * (1 - df['discount_pct'] / 100)

Menyimpan data CSV yang telah di bersihkan

In [28]:
# Laporan ringkas sebelum simpan
print("=" * 40)
print("RINGKASAN DATA SETELAH CLEANING")
print("=" * 40)
print(f"Jumlah baris  : {len(df)}")
print(f"Jumlah kolom  : {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplikat      : {df.duplicated().sum()}")
print(f"Periode data  : {df['order_date'].min().date()} s/d {df['order_date'].max().date()}")
print("=" * 40)
df.to_csv('../data/processed/ecommerce_cleaned_data.csv', index=False)

RINGKASAN DATA SETELAH CLEANING
Jumlah baris  : 1468
Jumlah kolom  : 15
Missing values: 0
Duplikat      : 0
Periode data  : 2024-01-06 s/d 2025-12-05
